In [1]:
import sys

sys.path.append("/om2/user/bjmedina/memory/utils")
from util_distance_models import *
from util_stimuli import *

In [5]:
# activations to load
model_name = "resnet50_audioset"
layer = "layer4"
stim_type = "NaturalSounds"
filename = f'/om2/user/bjmedina/memory/features/{model_name}-{layer}-{sound_set}.pt'
data, stimlist = load(filename)
print(data.shape)

torch.Size([999, 91])


In [6]:
# human responses to experiments that contain this stimulus set
all_results = glob.glob(stimuli_to_responses[sound_set])
exps = []
results = []
for result in all_results:
    part_result = pd.read_csv(result)
    main_exp = part_result[part_result['stim_type'] == 'main']
    if len(main_exp) < 256:
        continue
    exps.append(main_exp)
    results.append(result)
ISIs = set(main_exp.isi.tolist())

In [7]:
model = DistanceModelTwoPhaseBurst(noise_level=0.001, 
                                   dimensionality=data.shape[1], 
                                   scale=1, 
                                   end_scale=0.01,
                                   age_cutoff=3,
                                   model=model_name, 
                                   layer=layer,
                                   stim_type=stim_type)
samples = 2

Directory '/om2/user/bjmedina/memory/model_results/IdealObserverDistanceTwoPhaseBurst/NaturalSounds/resnet50_audioset/layer4/distance_model_start1_end0.01_c3_dim91/noise-0.001' already exists.


In [61]:
if exists(model.to_save_dir + "/observer_object.pkl"):
    
    with (open(model.to_save_dir + "/observer_object.pkl", "rb")) as openfile:
        model = pickle.load(openfile)
    
        print("Found file already. Loading it")
    
else:
    print("Created new object")


i = 0

for exp, result in zip(exps, results):
    print(result)
    
    for sample in range(1, samples+1): 
        model_df_save_dir = "{}/{}-{}".format(model.to_save_dir, sample, result.split('/')[-1])

        if exists(model_df_save_dir):
            continue

        model_resp, true_response, model_results = model.doTask(exp,seed=i, verbose=False)

        model_results.to_csv(model_df_save_dir)

        i += 1

Found file already. Loading it
/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/fktg4xbp414_natsounds_v1.csv
/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/nwyrngxky2g_natsounds_v1.csv
/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/7q407ssrcvs_natsounds_v1.csv


KeyboardInterrupt: 

In [51]:
ID = exp['sequence_file']
exp.columns

Index(['success', 'timeout', 'failed_images', 'failed_audio', 'failed_video',
       'trial_type', 'trial_index', 'time_elapsed', 'internal_node_id',
       'sequence_file', 'pay', 'passed_hc', 'completion_code', 'view_history',
       'rt', 'response', 'question_order', 'stimulus', 'stim_type',
       'hc_answer', 'correct_hc', 'repeat', 'yt_id', 'catch', 'isi',
       'is_correct', 'response_type'],
      dtype='object')

In [10]:
data.shape

torch.Size([999, 91])